## Data Cleaning
Bu dosyada Online Retail veri seti temizlenmiş, hatalı ve eksik kayıtlar
çıkarılarak analiz için hazır hale getirilmiştir.


## Çalışma Ortamının Hazırlanması

Bu adımda Google Colab ortamı Google Drive ile entegre edilmiştir.
Proje dizinine geçilerek dosya yapısı kontrol edilmiştir.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/online-retail-analysis')
os.listdir()


Mounted at /content/drive


['online-retail.zip',
 'data',
 'notebooks',
 'kaggle (1).json',
 'online-retail',
 'kaggle (1) (1).json']

## Gerekli Kütüphanelerin Kurulumu

Bu adımda veri setinin Kaggle üzerinden indirilebilmesi için Kaggle API kurulmuştur.


In [2]:
!pip install kaggle


## Kaggle API Anahtarının Yüklenmesi

Bu adımda Kaggle veri setine erişim sağlamak için gerekli olan `kaggle.json` dosyası Colab ortamına yüklenmiştir.


In [3]:
from google.colab import files
files.upload()


Saving kaggle (1).json to kaggle (1) (2).json


{'kaggle (1) (2).json': b'{"username":"ayadonduran","key":"31b6945db96b4514082328662b9f3aff"}'}

## Kaggle API Yapılandırması

Bu adımda Kaggle API anahtar dosyası ilgili dizine taşınmış ve
gerekli dosya izinleri ayarlanarak güvenli erişim sağlanmıştır.


In [4]:
!mkdir -p /root/.kaggle
!cp "/content/kaggle (1).json" /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json


cp: cannot stat '/content/kaggle (1).json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


## Veri Setinin İndirilmesi

Bu adımda analizde kullanılacak olan Online Retail veri seti
Kaggle üzerinden indirilmiştir.


In [5]:
!kaggle datasets download -d tunguz/online-retail


Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/cli.py", line 68, in main
    out = args.func(**command_args)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 1741, in dataset_download_cli
    with self.build_kaggle_client() as kaggle:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 688, in build_kaggle_client
    username=self.config_values['username'],
             ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'username'


## Veri Setinin Çıkarılması

İndirilen veri seti sıkıştırılmış dosyadan çıkarılarak
analiz için kullanılabilir hale getirilmiştir.


In [6]:
!unzip online-retail.zip -d ./online-retail


Archive:  online-retail.zip
replace ./online-retail/Online_Retail.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ./online-retail/Online_Retail.csv  


## Verinin Yüklenmesi ve İlk İnceleme

Bu adımda Online Retail veri seti pandas kullanılarak yüklenmiş
ve veri setinin genel yapısını görmek amacıyla ilk gözlemler yapılmıştır.


In [7]:
import pandas as pd

df = pd.read_csv('online-retail/Online_Retail.csv', encoding='ISO-8859-1')
df.head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


### Veri Setinin Genel Yapısı

Veri seti 541.909 gözlem ve 8 değişkenden oluşmaktadır. Değişkenlerin büyük bölümü
`object` tipindedir. `InvoiceDate` değişkeni tarih bilgisi içermesine rağmen
`object` tipindedir ve veri temizleme aşamasında `datetime` formatına dönüştürülecektir.

`CustomerID` sütununda yaklaşık 135.000 adet eksik değer bulunmaktadır.
Bu eksik kayıtlar, müşteri bazlı analizlerde (RFM gibi) sorun yaratacağından
ilerleyen adımlarda ele alınacaktır. Ayrıca `Description` sütununda da sınırlı
sayıda eksik değer bulunmaktadır.

Bu çıktıya göre veri temizleme sürecinde eksik değer analizi ve veri tipi
dönüşümleri yapılması gerekmektedir.


In [9]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


### Sayısal Değişkenlerin İstatistiksel Özeti

`Quantity` ve `UnitPrice` değişkenlerinde negatif minimum değerler
bulunmaktadır. Bu durum, iade işlemlerini veya hatalı kayıtları
işaret etmektedir. Özellikle `Quantity` değişkenindeki çok yüksek
standart sapma ve uç değerler veri setinde aykırı değerler
bulunduğunu göstermektedir.

`UnitPrice` sütununda da negatif ve aşırı yüksek değerler yer almaktadır.
Bu kayıtlar gerçek satış davranışını yansıtmadığı için analiz öncesinde
temizlenmesi gerekmektedir.

`CustomerID` yalnızca tanımlayıcı bir değişken olduğundan istatistiksel
yorum açısından sınırlı bilgi sunmaktadır; ancak eksik değer problemi
bulunduğu daha önce tespit edilmiştir.


In [10]:
df.isnull().sum()

,0
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


### Eksik Değer Analizi

Veri setinde en fazla eksik değer `CustomerID` sütununda bulunmaktadır.
Bu sütun müşteri bazlı analizler (RFM analizi gibi) için kritik
öneme sahip olduğundan, `CustomerID` bilgisi olmayan kayıtlar
müşteri segmentasyonu çalışmalarında kullanılamaz.

`Description` sütununda bulunan eksik değerler sınırlı sayıdadır.
Bu sütun ürün açıklaması içerdiğinden, analiz açısından
ikincil öneme sahiptir.

Diğer sütunlarda eksik değer bulunmamaktadır. Bu nedenle
veri temizleme aşamasında temel odak noktası
`CustomerID` eksikliği olacaktır.


In [11]:
df.nunique()

,0
InvoiceNo,25900
StockCode,4070
Description,4223
Quantity,722
InvoiceDate,23260
UnitPrice,1630
CustomerID,4372
Country,38


### Benzersiz Değer Analizi

Veri setinde toplam **25.900** farklı fatura (`InvoiceNo`) bulunmaktadır.
Bu durum, bir faturada birden fazla ürün satırının yer aldığını göstermektedir.

Toplam **4.372** benzersiz müşteri (`CustomerID`) bulunmaktadır.
Bu bilgi, müşteri segmentasyonu ve RFM analizi için yeterli
bir müşteri çeşitliliği olduğunu göstermektedir.

Ürün bazında incelendiğinde **4.070** farklı ürün kodu (`StockCode`)
ve **4.223** farklı ürün açıklaması (`Description`) yer almaktadır.

Veri seti **38** farklı ülkeyi kapsamaktadır.
Bu durum, çok uluslu müşteri davranışlarının analiz edilmesine
olanak tanımaktadır.


In [12]:
df['Country'].value_counts().head()


,count
Country,
United Kingdom,495478
Germany,9495
France,8557
EIRE,8196
Spain,2533


### Ülke Bazlı Dağılım

Veri setindeki işlemlerin büyük çoğunluğu **Birleşik Krallık (United Kingdom)** kaynaklıdır.
Bu durum, veri setinin merkezinin İngiltere olduğunu göstermektedir.

Diğer ülkeler (Almanya, Fransa, İrlanda, İspanya) daha düşük işlem sayısına sahiptir.
Bu nedenle analizlerin büyük kısmı İngiltere müşteri davranışlarını
yansıtmaktadır.


In [13]:
df['InvoiceDate'] = pd.to_datetime(
    df['InvoiceDate'],
    format='%m/%d/%y %H:%M',
    errors='coerce'
)

df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')

df.groupby('InvoiceMonth').size().head()


,0
InvoiceMonth,
2010-12,42481
2011-01,35147
2011-02,27707
2011-03,36748
2011-04,29916


### Zaman Değişkeninin Dönüştürülmesi

`InvoiceDate` değişkeni tarih–zaman formatına dönüştürülmüştür.
Bu sayede aylık, dönemsel ve zamana bağlı analizlerin yapılabilmesi sağlanmıştır.

Oluşturulan `InvoiceMonth` değişkeni ile işlem sayılarının aylara göre dağılımı
incelenmiştir. İlk gözlemler, işlemlerin belirli aylarda yoğunlaştığını
göstermektedir.


#Veri Temizleme

In [14]:
df = df.dropna(subset=['CustomerID'])


### Eksik Müşteri Kimliklerinin Temizlenmesi

RFM analizi ve müşteri bazlı segmentasyon çalışmalarında
`CustomerID` bilgisi zorunlu olduğundan,
bu değişkende eksik değere sahip gözlemler veri setinden çıkarılmıştır.

Bu işlem sonrasında veri seti, müşteri odaklı analizler için
daha tutarlı ve analiz edilebilir hale getirilmiştir.


In [15]:
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]


### İade (Cancellation) İşlemlerinin Çıkarılması

Veri setinde `InvoiceNo` değeri **"C"** ile başlayan kayıtlar,
iptal veya iade edilen işlemleri temsil etmektedir.

Ciro, müşteri değeri (RFM) ve satış odaklı analizlerin
yanıltıcı olmaması için bu tür işlemler veri setinden çıkarılmıştır.


In [16]:
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]


### Negatif ve Sıfır Değerlerin Temizlenmesi

Veri setinde negatif veya sıfır değerli `Quantity` ve `UnitPrice` kayıtları,
gerçek satış işlemlerini temsil etmemektedir.

Bu tür değerler:
- İade işlemlerinden kalan hatalı kayıtlar olabilir
- Ciro ve müşteri analizlerini yanıltır

Bu nedenle yalnızca **pozitif miktar** ve **pozitif birim fiyat**
içeren işlemler analiz kapsamına alınmıştır.


In [17]:
df.isnull().sum()



,0
InvoiceNo,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,0
Country,0
InvoiceMonth,0


### Eksik Değer Kontrolü

Veri temizleme adımlarının ardından yapılan kontrolde,
analiz kapsamında kullanılan tüm değişkenlerde eksik değer
kalmadığı görülmüştür.

Bu durum:
- Ciro hesaplamaları
- Zaman bazlı analizler
- RFM ve segmentasyon çalışmaları

için veri setinin analiz aşamasına tamamen hazır olduğunu
göstermektedir.


In [18]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,397884.000000,397884,397884.000000,397884.000000
mean,12.988238,2011-07-10 23:41:23.511023360,3.116488,15294.423453
min,1.000000,2010-12-01 08:26:00,0.001000,12346.000000
25%,2.000000,2011-04-07 11:12:00,1.250000,13969.000000
50%,6.000000,2011-07-31 14:39:00,1.950000,15159.000000
75%,12.000000,2011-10-20 14:33:00,3.750000,16795.000000
max,80995.000000,2011-12-09 12:50:00,8142.750000,18287.000000
std,179.331775,NaN,22.097877,1713.141560


### Sayısal Değişkenlerin İstatistiksel Özeti

Veri temizleme sürecinde:
- Negatif `Quantity` ve `UnitPrice` değerleri
- İade işlemlerine ait kayıtlar

analiz dışı bırakılmıştır.

Bu nedenle mevcut istatistikler,
yalnızca **gerçek ve geçerli satış işlemlerini** temsil etmektedir.

Özet istatistikler incelendiğinde:
- `Quantity` ve `TotalPrice` değişkenlerinde yüksek maksimum değerler,
  bazı müşterilerin toplu alım yaptığını göstermektedir.
- Ortalama ve medyan değerler arasındaki farklar,
  satış dağılımının sağa çarpık olduğunu ortaya koymaktadır.

Bu yapı, müşteri segmentasyonu ve ciro analizleri için
verinin güvenilir ve analiz edilebilir olduğunu göstermektedir.


## Keşifsel Veri Analizi Sonrası Değerlendirme

Yapılan keşifsel analizler sonucunda:

- Veri setinin satış davranışlarını güvenilir şekilde temsil ettiği,
- Zaman bazlı ciro ve sipariş dağılımlarının belirgin eğilimler gösterdiği,
- Satışların müşteri bazında heterojen bir yapı sergilediği

tespit edilmiştir.

Bu noktadan itibaren, müşteri davranışlarını daha detaylı
inceleyebilmek ve işletme açısından anlamlı müşteri grupları
oluşturabilmek amacıyla **RFM (Recency, Frequency, Monetary) analizi**
uygulanmıştır.


#RFM Analizine Geçiş

## RFM Değişkenlerinin Hesaplanması

Bu adımda müşteri bazlı olarak:

- **Recency**: Müşterinin son alışverişinden bu yana geçen gün sayısı
- **Frequency**: Toplam alışveriş sayısı
- **Monetary**: Toplam harcama tutarı

hesaplanmıştır.

Referans tarih olarak veri setindeki en son işlem tarihinin
bir gün sonrası alınmıştır.


In [20]:
import datetime as dt

reference_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']


In [21]:
rfm.head()


,Recency,Frequency,Monetary
CustomerID,,,
12346.0,326,1,77183.60
12347.0,2,7,4310.00
12348.0,75,4,1797.24
12349.0,19,1,1757.55
12350.0,310,1,334.40


### RFM Tablosunun İlk Gözlemi

Oluşturulan RFM tablosu, her satırda bir müşteriyi temsil etmektedir.
Müşteri bazında:

- `Recency`: Son alışverişten bu yana geçen gün sayısı
- `Frequency`: Toplam alışveriş sayısı
- `Monetary`: Toplam harcama tutarı

metriklerinin başarıyla hesaplandığı görülmektedir.

Değerler arasındaki farklılıklar, müşteri davranışlarının
homojen olmadığını ve segmentasyon için uygun bir yapı
oluşturduğunu göstermektedir.


## RFM Skorlarının Oluşturulması

Bu adımda RFM metrikleri, müşteriler arasında karşılaştırılabilir
hale getirilmek amacıyla 5 eşit parçaya (quantile) bölünmüştür.

- **Recency** için daha düşük gün sayısı daha iyi olduğu için
  ters sıralı skorlar atanmıştır.
- **Frequency** değişkeninde tekrar eden değerler nedeniyle
  sıralama problemi yaşamamak adına `rank(method='first')` kullanılmıştır.
- **Monetary** için yüksek harcama daha yüksek skor ile
  temsil edilmiştir.


In [22]:
rfm['R_Score'] = pd.qcut(
    rfm['Recency'],
    5,
    labels=[5, 4, 3, 2, 1]
)

rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm['M_Score'] = pd.qcut(
    rfm['Monetary'],
    5,
    labels=[1, 2, 3, 4, 5]
)


In [23]:
rfm[['R_Score', 'F_Score', 'M_Score']].isnull().sum()


,0
R_Score,0
F_Score,0
M_Score,0


### RFM Skorlarında Eksik Değer Kontrolü

R, F ve M skorları oluşturulduktan sonra yapılan kontrolde,
herhangi bir eksik değer bulunmadığı görülmüştür.

Bu durum, quantile tabanlı skorlamanın tüm müşteriler
için başarıyla uygulanabildiğini göstermektedir.


In [24]:
rfm['RFM_Score'] = (
    rfm['R_Score'].astype(str) +
    rfm['F_Score'].astype(str) +
    rfm['M_Score'].astype(str)
)


### RFM Skorlarının Birleştirilmesi

Her müşteri için hesaplanan R, F ve M skorları birleştirilerek
üç haneli bir **RFM skoru** oluşturulmuştur.

Bu skor, müşterilerin:
- Güncellik
- Satın alma sıklığı
- Harcama düzeyi

açısından genel davranış profilini temsil etmektedir ve
segmentasyon işlemlerinin temelini oluşturmaktadır.


In [25]:
rfm[['R_Score', 'F_Score', 'M_Score', 'RFM_Score']].head()


,R_Score,F_Score,M_Score,RFM_Score
CustomerID,,,,
12346.0,1,1,5,115
12347.0,5,5,5,555
12348.0,2,4,4,244
12349.0,4,1,4,414
12350.0,1,1,2,112


### RFM Skorlarının İlk Gözlemi

R, F ve M skorlarının birleştirilmesiyle oluşturulan
`RFM_Score` değişkeninin, müşteri davranışlarını
ayırt edici şekilde temsil ettiği görülmektedir.

Örneğin:
- **555** skoruna sahip müşteriler hem yakın zamanda alışveriş yapmış,
  hem sık alışveriş yapan hem de yüksek harcama düzeyine sahiptir.
- **115** gibi skorlar, uzun süredir alışveriş yapmamış ancak
  geçmişte yüksek harcama gerçekleştirmiş müşterileri temsil edebilir.

Bu yapı, anlamlı müşteri segmentleri oluşturmak için
uygun bir temel sunmaktadır.


In [26]:
rfm['RFM_Score'].nunique()


118

### RFM Skorlarının Çeşitliliği

Oluşturulan RFM skorlarının 118 farklı kombinasyona sahip olduğu
görülmektedir.

Bu durum, müşteri davranışlarının tek tip olmadığını ve
segmentasyon için yeterli düzeyde ayrışma bulunduğunu
göstermektedir.


## Müşteri Segmentlerinin Tanımlanması

Bu adımda, R ve F skorlarının kombinasyonları kullanılarak
müşteriler iş açısından anlamlı segmentlere ayrılmıştır.

Segmentler, literatürde yaygın olarak kullanılan RFM
segmentasyon yaklaşımına göre tanımlanmıştır.

Amaç:
- Değerli müşterileri (Champions, Loyal Customers) belirlemek
- Kaybedilme riski taşıyan müşterileri (At Risk, Cannot Lose) tespit etmek
- Yeni ve potansiyel müşterileri ayrı gruplar halinde incelemektir


In [27]:
segment_map = {
    r'[4-5][4-5]': 'Champions',
    r'[3-4][3-5]': 'Loyal Customers',
    r'[4-5][1-3]': 'Potential Loyalists',
    r'[5][1]': 'New Customers',
    r'[3][1-2]': 'Promising',
    r'[2-3][2-3]': 'Need Attention',
    r'[2][1-2]': 'About To Sleep',
    r'[1-2][3-5]': 'At Risk',
    r'[1][4-5]': 'Cannot Lose',
    r'[1][1-2]': 'Lost'
}
rfm['Segment'] = (
    rfm['R_Score'].astype(str) +
    rfm['F_Score'].astype(str)
).replace(segment_map, regex=True)


In [28]:
rfm['Segment'].value_counts()


,count
Segment,
Champions,1139
Loyal Customers,685
Lost,664
Potential Loyalists,455
Need Attention,428
At Risk,417
Promising,351
About To Sleep,199


## RFM Segmentasyonu

Müşteriler RFM skorlarına göre segmentlere ayrılmıştır. En yüksek pay **Champions** ve **Loyal Customers** segmentlerine aittir. **Lost** ve **At Risk** grupları ise müşteri kaybı riskini göstermektedir.


In [29]:
rfm['Segment'].value_counts(normalize=True) * 100


,proportion
Segment,
Champions,26.256339
Loyal Customers,15.790687
Lost,15.306593
Potential Loyalists,10.488704
Need Attention,9.866298
At Risk,9.612725
Promising,8.091286
About To Sleep,4.587367


## Segment Dağılımı (%)

Müşterilerin %26,26’sı **Champions** segmentinde yer almaktadır. Bunu **Loyal Customers** (%15,79) ve **Lost** (%15,31) segmentleri izlemektedir. **At Risk** (%9,61) ve **Need Attention** (%9,87) grupları, müşteri kaybı açısından kritik öneme sahiptir.


In [30]:
rfm[rfm['Segment'].isnull()]


,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
CustomerID,,,,,,,,


## Segment Ataması Kontrolü

Segment sütununda boş (NaN) değer bulunmamaktadır. Tüm müşteriler RFM skorlarına göre başarıyla segmentlere atanmıştır.


## Temizlenmiş Verinin Kaydedilmesi


In [31]:
import os

os.makedirs('data/processed', exist_ok=True)

df.to_csv('data/processed/cleaned_online_retail.csv', index=False)

os.listdir('data/processed')


['cleaned_online_retail.csv', 'rfm_table.csv']